# Data Cleaning - DDI and Medications
This notebook cleans and preprocesses the raw data from v1_raw and writes cleaned versions to v2_clean.

**Input**:  
- `med-data/v1_raw/ddi/db_drug_interactions.parquet`  
- `med-data/v1_raw/medications/medications_combined.parquet`

**Output**:  
- `med-data/v2_clean/ddi/db_drug_interactions_clean.parquet`  
- `med-data/v2_clean/medications/medications_clean.parquet`

In [1]:
# Import dependencies

import os
import sys
import logging
import time
import re
from datetime import datetime
import numpy as np
import pandas as pd
import s3fs
import pyarrow as pa
from importlib.metadata import version
from config import *

In [2]:
# Verify dependencies

def print_version():
    print("pandas:", pd.__version__)
    print("numpy:", np.__version__)
    print("s3fs:", s3fs.__version__)
    print("pyarrow:", pa.__version__)

print_version()

pandas: 2.3.3
numpy: 2.3.4
s3fs: 2025.10.0
pyarrow: 22.0.0


In [3]:
# Set up logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logging.info("Logging configured successfully")

2025-11-27 10:02:05,121 INFO Logging configured successfully


In [4]:
# Load configuration

logging.info(f"MinIO endpoint: {MINIO_ENDPOINT}")
logging.info(f"Source: {DEST_BUCKET}/v1_raw/")
logging.info(f"Destination: {DEST_BUCKET}/v2_clean/")

2025-11-27 10:02:09,087 INFO MinIO endpoint: localhost:9000
2025-11-27 10:02:09,088 INFO Source: med-data/v1_raw/
2025-11-27 10:02:09,088 INFO Destination: med-data/v2_clean/


In [5]:
# Create S3FileSystem for MinIO

logging.info(f"Initializing S3FileSystem for MinIO at {MINIO_ENDPOINT}")
fs = s3fs.S3FileSystem(
    anon=False,
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    client_kwargs={'endpoint_url': f"http://{MINIO_ENDPOINT}"}
)
logging.info("S3FileSystem created successfully")

2025-11-27 10:02:13,620 INFO Initializing S3FileSystem for MinIO at localhost:9000
2025-11-27 10:02:13,622 INFO S3FileSystem created successfully


---
## Part 1: Load Raw Data

In [6]:
# Load DDI reference dataset from v1_raw

ddi_uri = f"s3://{DEST_BUCKET}/{V1_RAW_DDI_PREFIX}db_drug_interactions.parquet"
logging.info(f"Reading DDI data: {ddi_uri}")

start_time = time.time()
df_ddi_raw = pd.read_parquet(ddi_uri, filesystem=fs)
elapsed = time.time() - start_time

logging.info(f"Loaded {len(df_ddi_raw):,} DDI records in {elapsed:.2f}s")

print(f"\nDDI Raw Data Shape: {df_ddi_raw.shape}")
df_ddi_raw.head()

2025-11-27 10:02:21,481 INFO Reading DDI data: s3://med-data/v1_raw/ddi/db_drug_interactions.parquet
2025-11-27 10:02:21,641 INFO Loaded 191,541 DDI records in 0.16s



DDI Raw Data Shape: (191541, 3)


,Drug 1,Drug 2,Interaction Description
0,Trioxsalen,Verteporfin,Trioxsalen may increase the photosensitizing activities of Verteporfin.
1,Aminolevulinic acid,Verteporfin,Aminolevulinic acid may increase the photosensitizing activities of Verteporfin.
2,Titanium dioxide,Verteporfin,Titanium dioxide may increase the photosensitizing activities of Verteporfin.
3,Tiaprofenic acid,Verteporfin,Tiaprofenic acid may increase the photosensitizing activities of Verteporfin.
4,Cyamemazine,Verteporfin,Cyamemazine may increase the photosensitizing activities of Verteporfin.


In [7]:
# Load medications dataset from v1_raw

meds_uri = f"s3://{DEST_BUCKET}/{V1_RAW_MEDICATIONS_PREFIX}medications_combined.parquet"
logging.info(f"Reading medications data: {meds_uri}")

start_time = time.time()
df_meds_raw = pd.read_parquet(meds_uri, filesystem=fs)
elapsed = time.time() - start_time

logging.info(f"Loaded {len(df_meds_raw):,} medication records in {elapsed:.2f}s")

print(f"\nMedications Raw Data Shape: {df_meds_raw.shape}")
df_meds_raw.head()

2025-11-27 10:02:25,463 INFO Reading medications data: s3://med-data/v1_raw/medications/medications_combined.parquet
2025-11-27 10:02:25,492 INFO Loaded 36 medication records in 0.03s



Medications Raw Data Shape: (36, 18)


,PatientSID,PatientIEN,Sta3n,DrugNameWithoutDose,DrugNameWithDose,SourceSystem,MedicationDateTime,StartDate,EndDate,Status,DaysSupply,Quantity,DEASchedule,ControlledSubstanceFlag,OrderNumber,ProviderSID,LocalDrugSID,NationalDrugSID
0,1001,PtIEN1001,508,LISINOPRIL,LISINOPRIL 10MG TAB,BCMA,2025-01-01 08:05:00,2025-01-01 08:05:00,NaT,GIVEN,NaN,NaN,None,None,IP-2025-001001,1001,10002,20002
1,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,BCMA,2025-01-01 12:10:00,2025-01-01 12:10:00,NaT,GIVEN,NaN,NaN,None,None,IP-2025-001002,1001,10001,20001
2,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,BCMA,2025-01-01 18:08:00,2025-01-01 18:08:00,NaT,GIVEN,NaN,NaN,None,None,IP-2025-001002,1001,10001,20001
3,1001,PtIEN1001,508,LISINOPRIL,LISINOPRIL 10MG TAB,BCMA,2025-01-02 08:45:00,2025-01-02 08:45:00,NaT,GIVEN,NaN,NaN,None,None,IP-2025-001001,1001,10002,20002
4,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,RxOut,2025-01-15 10:30:00,2025-01-15 10:30:00,2025-01-15,ACTIVE,90.0,180.0,None,N,2024-001-0001,1001,10001,20001


---
## Part 2: Clean DDI Reference Dataset

In [8]:
# Initial data quality assessment for DDI dataset

print("="*80)
print("DDI DATASET - INITIAL QUALITY ASSESSMENT")
print("="*80)

print(f"\nShape: {df_ddi_raw.shape}")
print(f"\nMissing values:")
print(df_ddi_raw.isnull().sum())

print(f"\nDuplicate rows: {df_ddi_raw.duplicated().sum()}")

print(f"\nData types:")
print(df_ddi_raw.dtypes)

print("="*80)

DDI DATASET - INITIAL QUALITY ASSESSMENT

Shape: (191541, 3)

Missing values:
Drug 1                     0
Drug 2                     0
Interaction Description    0
dtype: int64

Duplicate rows: 0

Data types:
Drug 1                     object
Drug 2                     object
Interaction Description    object
dtype: object


In [9]:
# Clean DDI dataset

logging.info("Cleaning DDI dataset...")

# Create copy for cleaning
df_ddi_clean = df_ddi_raw.copy()

initial_count = len(df_ddi_clean)
logging.info(f"Starting with {initial_count:,} records")

# 1. Remove duplicates
df_ddi_clean = df_ddi_clean.drop_duplicates()
duplicates_removed = initial_count - len(df_ddi_clean)
logging.info(f"Removed {duplicates_removed:,} duplicate records")

# 2. Remove records with missing values
before_nulls = len(df_ddi_clean)
df_ddi_clean = df_ddi_clean.dropna()
nulls_removed = before_nulls - len(df_ddi_clean)
logging.info(f"Removed {nulls_removed:,} records with missing values")

# 3. Strip whitespace from string columns
df_ddi_clean['Drug 1'] = df_ddi_clean['Drug 1'].str.strip()
df_ddi_clean['Drug 2'] = df_ddi_clean['Drug 2'].str.strip()
df_ddi_clean['Interaction Description'] = df_ddi_clean['Interaction Description'].str.strip()
logging.info("Stripped whitespace from string columns")

# 4. Remove empty strings
before_empty = len(df_ddi_clean)
df_ddi_clean = df_ddi_clean[
    (df_ddi_clean['Drug 1'] != '') & 
    (df_ddi_clean['Drug 2'] != '') & 
    (df_ddi_clean['Interaction Description'] != '')
]
empty_removed = before_empty - len(df_ddi_clean)
logging.info(f"Removed {empty_removed:,} records with empty strings")

# 5. Add normalized drug name columns (for matching)
def normalize_drug_name(drug_name):
    """Normalize drug name: uppercase, remove salt suffixes."""
    if pd.isna(drug_name):
        return None
    name = str(drug_name).upper().strip()
    suffixes = [' HCL', ' HYDROCHLORIDE', ' SODIUM', ' POTASSIUM', ' CALCIUM',
                ' SULFATE', ' TARTRATE', ' SUCCINATE', ' MALEATE', ' FUMARATE',
                ' ACETATE', ' CITRATE', ' PHOSPHATE']
    for suffix in suffixes:
        if name.endswith(suffix):
            name = name[:-len(suffix)].strip()
            break
    return name

df_ddi_clean['Drug1_Normalized'] = df_ddi_clean['Drug 1'].apply(normalize_drug_name)
df_ddi_clean['Drug2_Normalized'] = df_ddi_clean['Drug 2'].apply(normalize_drug_name)
logging.info("Added normalized drug name columns")

# 6. Extract severity classification from interaction description
def classify_severity(description):
    """Classify interaction severity based on keywords."""
    if pd.isna(description):
        return 'Unknown'
    desc_lower = description.lower()
    if any(word in desc_lower for word in ['contraindicated', 'avoid', 'serious', 'severe']):
        return 'High'
    elif any(word in desc_lower for word in ['caution', 'monitor', 'may increase', 'may decrease']):
        return 'Moderate'
    else:
        return 'Low'

df_ddi_clean['Severity'] = df_ddi_clean['Interaction Description'].apply(classify_severity)
logging.info("Added severity classification column")

logging.info(f"DDI cleaning complete: {len(df_ddi_clean):,} records")

print(f"\nCleaned DDI Data Shape: {df_ddi_clean.shape}")
df_ddi_clean.head()

2025-11-27 10:02:42,832 INFO Cleaning DDI dataset...
2025-11-27 10:02:42,837 INFO Starting with 191,541 records
2025-11-27 10:02:42,898 INFO Removed 0 duplicate records
2025-11-27 10:02:42,910 INFO Removed 0 records with missing values
2025-11-27 10:02:42,939 INFO Stripped whitespace from string columns
2025-11-27 10:02:42,960 INFO Removed 0 records with empty strings
2025-11-27 10:02:43,190 INFO Added normalized drug name columns
2025-11-27 10:02:43,343 INFO Added severity classification column
2025-11-27 10:02:43,344 INFO DDI cleaning complete: 191,541 records



Cleaned DDI Data Shape: (191541, 6)


,Drug 1,Drug 2,Interaction Description,Drug1_Normalized,Drug2_Normalized,Severity
0,Trioxsalen,Verteporfin,Trioxsalen may increase the photosensitizing activities of Verteporfin.,TRIOXSALEN,VERTEPORFIN,Moderate
1,Aminolevulinic acid,Verteporfin,Aminolevulinic acid may increase the photosensitizing activities of Verteporfin.,AMINOLEVULINIC ACID,VERTEPORFIN,Moderate
2,Titanium dioxide,Verteporfin,Titanium dioxide may increase the photosensitizing activities of Verteporfin.,TITANIUM DIOXIDE,VERTEPORFIN,Moderate
3,Tiaprofenic acid,Verteporfin,Tiaprofenic acid may increase the photosensitizing activities of Verteporfin.,TIAPROFENIC ACID,VERTEPORFIN,Moderate
4,Cyamemazine,Verteporfin,Cyamemazine may increase the photosensitizing activities of Verteporfin.,CYAMEMAZINE,VERTEPORFIN,Moderate


In [10]:
# DDI cleaning summary

print("\n" + "="*80)
print("DDI DATASET - CLEANING SUMMARY")
print("="*80)

print(f"Original records:     {initial_count:,}")
print(f"Cleaned records:      {len(df_ddi_clean):,}")
print(f"Records removed:      {initial_count - len(df_ddi_clean):,}")
print(f"Removal rate:         {((initial_count - len(df_ddi_clean)) / initial_count * 100):.2f}%")

print(f"\nSeverity distribution:")
print(df_ddi_clean['Severity'].value_counts())

print(f"\nNew columns added:")
print("  - Drug1_Normalized")
print("  - Drug2_Normalized")
print("  - Severity")

print("="*80)


DDI DATASET - CLEANING SUMMARY
Original records:     191,541
Cleaned records:      191,541
Records removed:      0
Removal rate:         0.00%

Severity distribution:
Severity
Low         144313
Moderate     47228
Name: count, dtype: int64

New columns added:
  - Drug1_Normalized
  - Drug2_Normalized
  - Severity


---
## Part 3: Clean Medications Dataset

In [11]:
# Initial data quality assessment for medications dataset

print("="*80)
print("MEDICATIONS DATASET - INITIAL QUALITY ASSESSMENT")
print("="*80)

print(f"\nShape: {df_meds_raw.shape}")
print(f"\nMissing values:")
print(df_meds_raw.isnull().sum())

print(f"\nDuplicate rows: {df_meds_raw.duplicated().sum()}")

print(f"\nData types:")
print(df_meds_raw.dtypes)

print(f"\nSource system distribution:")
print(df_meds_raw['SourceSystem'].value_counts())

print("="*80)

MEDICATIONS DATASET - INITIAL QUALITY ASSESSMENT

Shape: (36, 18)

Missing values:
PatientSID                  0
PatientIEN                  0
Sta3n                       0
DrugNameWithoutDose         0
DrugNameWithDose            0
SourceSystem                0
MedicationDateTime          0
StartDate                   0
EndDate                    17
Status                      0
DaysSupply                 17
Quantity                   17
DEASchedule                34
ControlledSubstanceFlag    19
OrderNumber                 0
ProviderSID                 0
LocalDrugSID                0
NationalDrugSID             0
dtype: int64

Duplicate rows: 0

Data types:
PatientSID                          int64
PatientIEN                         object
Sta3n                               int64
DrugNameWithoutDose                object
DrugNameWithDose                   object
SourceSystem                       object
MedicationDateTime         datetime64[ns]
StartDate                  datetime64[

In [12]:
# Clean medications dataset

logging.info("Cleaning medications dataset...")

# Create copy for cleaning
df_meds_clean = df_meds_raw.copy()

initial_count = len(df_meds_clean)
logging.info(f"Starting with {initial_count:,} records")

# 1. Remove exact duplicates
df_meds_clean = df_meds_clean.drop_duplicates()
duplicates_removed = initial_count - len(df_meds_clean)
logging.info(f"Removed {duplicates_removed:,} exact duplicate records")

# 2. Remove records with missing critical fields
before_critical = len(df_meds_clean)
df_meds_clean = df_meds_clean.dropna(subset=['PatientSID', 'DrugNameWithoutDose', 'MedicationDateTime'])
critical_removed = before_critical - len(df_meds_clean)
logging.info(f"Removed {critical_removed:,} records with missing critical fields")

# 3. Ensure proper data types
df_meds_clean['PatientSID'] = df_meds_clean['PatientSID'].astype('int64')
df_meds_clean['MedicationDateTime'] = pd.to_datetime(df_meds_clean['MedicationDateTime'])
df_meds_clean['StartDate'] = pd.to_datetime(df_meds_clean['StartDate'])
logging.info("Ensured proper data types")

# 4. Strip whitespace from string columns
string_cols = ['DrugNameWithoutDose', 'DrugNameWithDose', 'SourceSystem', 'Status']
for col in string_cols:
    if col in df_meds_clean.columns:
        df_meds_clean[col] = df_meds_clean[col].astype(str).str.strip()
logging.info("Stripped whitespace from string columns")

# 5. Add normalized drug name column
def extract_base_drug_name(drug_name):
    """Extract base drug name: remove dose, normalize."""
    if pd.isna(drug_name):
        return None
    name = str(drug_name).upper().strip()
    # Remove dose info (everything after first digit)
    name = re.split(r'\s*\d', name)[0].strip()
    # Apply normalization
    return normalize_drug_name(name)

df_meds_clean['DrugName_Normalized'] = df_meds_clean['DrugNameWithoutDose'].apply(extract_base_drug_name)
logging.info("Added normalized drug name column")

# 6. Add date components for analysis
df_meds_clean['MedicationYear'] = df_meds_clean['MedicationDateTime'].dt.year
df_meds_clean['MedicationMonth'] = df_meds_clean['MedicationDateTime'].dt.month
df_meds_clean['MedicationDayOfWeek'] = df_meds_clean['MedicationDateTime'].dt.dayofweek
df_meds_clean['MedicationHour'] = df_meds_clean['MedicationDateTime'].dt.hour
logging.info("Added date component columns")

# 7. Sort by patient and datetime
df_meds_clean = df_meds_clean.sort_values(['PatientSID', 'MedicationDateTime'])
logging.info("Sorted by patient and datetime")

# 8. Reset index
df_meds_clean = df_meds_clean.reset_index(drop=True)

logging.info(f"Medications cleaning complete: {len(df_meds_clean):,} records")

print(f"\nCleaned Medications Data Shape: {df_meds_clean.shape}")
df_meds_clean.head()

2025-11-27 10:03:34,006 INFO Cleaning medications dataset...
2025-11-27 10:03:34,007 INFO Starting with 36 records
2025-11-27 10:03:34,009 INFO Removed 0 exact duplicate records
2025-11-27 10:03:34,011 INFO Removed 0 records with missing critical fields
2025-11-27 10:03:34,012 INFO Ensured proper data types
2025-11-27 10:03:34,013 INFO Stripped whitespace from string columns
2025-11-27 10:03:34,014 INFO Added normalized drug name column
2025-11-27 10:03:34,015 INFO Added date component columns
2025-11-27 10:03:34,017 INFO Sorted by patient and datetime
2025-11-27 10:03:34,017 INFO Medications cleaning complete: 36 records



Cleaned Medications Data Shape: (36, 23)


,PatientSID,PatientIEN,Sta3n,DrugNameWithoutDose,DrugNameWithDose,SourceSystem,MedicationDateTime,StartDate,EndDate,Status,...,ControlledSubstanceFlag,OrderNumber,ProviderSID,LocalDrugSID,NationalDrugSID,DrugName_Normalized,MedicationYear,MedicationMonth,MedicationDayOfWeek,MedicationHour
0,1001,PtIEN1001,508,LISINOPRIL,LISINOPRIL 10MG TAB,BCMA,2025-01-01 08:05:00,2025-01-01 08:05:00,NaT,GIVEN,...,None,IP-2025-001001,1001,10002,20002,LISINOPRIL,2025,1,2,8
1,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,BCMA,2025-01-01 12:10:00,2025-01-01 12:10:00,NaT,GIVEN,...,None,IP-2025-001002,1001,10001,20001,METFORMIN,2025,1,2,12
2,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,BCMA,2025-01-01 18:08:00,2025-01-01 18:08:00,NaT,GIVEN,...,None,IP-2025-001002,1001,10001,20001,METFORMIN,2025,1,2,18
3,1001,PtIEN1001,508,LISINOPRIL,LISINOPRIL 10MG TAB,BCMA,2025-01-02 08:45:00,2025-01-02 08:45:00,NaT,GIVEN,...,None,IP-2025-001001,1001,10002,20002,LISINOPRIL,2025,1,3,8
4,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,RxOut,2025-01-15 10:30:00,2025-01-15 10:30:00,2025-01-15,ACTIVE,...,N,2024-001-0001,1001,10001,20001,METFORMIN,2025,1,2,10


In [13]:
# Medications cleaning summary

print("\n" + "="*80)
print("MEDICATIONS DATASET - CLEANING SUMMARY")
print("="*80)

print(f"Original records:     {initial_count:,}")
print(f"Cleaned records:      {len(df_meds_clean):,}")
print(f"Records removed:      {initial_count - len(df_meds_clean):,}")
print(f"Removal rate:         {((initial_count - len(df_meds_clean)) / initial_count * 100):.2f}%")

print(f"\nUnique patients:      {df_meds_clean['PatientSID'].nunique()}")
print(f"Unique medications:   {df_meds_clean['DrugName_Normalized'].nunique()}")

print(f"\nSource system distribution:")
print(df_meds_clean['SourceSystem'].value_counts())

print(f"\nDate range:")
print(f"  Earliest: {df_meds_clean['MedicationDateTime'].min()}")
print(f"  Latest:   {df_meds_clean['MedicationDateTime'].max()}")

print(f"\nNew columns added:")
print("  - DrugName_Normalized")
print("  - MedicationYear")
print("  - MedicationMonth")
print("  - MedicationDayOfWeek")
print("  - MedicationHour")

print("="*80)


MEDICATIONS DATASET - CLEANING SUMMARY
Original records:     36
Cleaned records:      36
Records removed:      0
Removal rate:         0.00%

Unique patients:      10
Unique medications:   21

Source system distribution:
SourceSystem
RxOut    19
BCMA     17
Name: count, dtype: int64

Date range:
  Earliest: 2025-01-01 08:05:00
  Latest:   2025-04-10 14:25:00

New columns added:
  - DrugName_Normalized
  - MedicationYear
  - MedicationMonth
  - MedicationDayOfWeek
  - MedicationHour


---
## Part 4: Write Cleaned Data to v2_clean

In [14]:
# Write cleaned DDI dataset to v2_clean

ddi_clean_filename = "db_drug_interactions_clean.parquet"
ddi_clean_uri = f"s3://{DEST_BUCKET}/{V2_CLEAN_DDI_PREFIX}{ddi_clean_filename}"
logging.info(f"Writing cleaned DDI data: {ddi_clean_uri}")

start_time = time.time()

df_ddi_clean.to_parquet(
    ddi_clean_uri,
    engine='pyarrow',
    filesystem=fs,
    compression='snappy',
    index=False
)

elapsed = time.time() - start_time
logging.info(f"Successfully wrote {len(df_ddi_clean):,} records in {elapsed:.2f}s")

print(f"✓ DDI clean data written to: {ddi_clean_uri}")

2025-11-27 10:04:00,995 INFO Writing cleaned DDI data: s3://med-data/v2_clean/ddi/db_drug_interactions_clean.parquet
2025-11-27 10:04:01,091 INFO Successfully wrote 191,541 records in 0.09s


✓ DDI clean data written to: s3://med-data/v2_clean/ddi/db_drug_interactions_clean.parquet


In [15]:
# Write cleaned medications dataset to v2_clean

meds_clean_filename = "medications_clean.parquet"
meds_clean_uri = f"s3://{DEST_BUCKET}/{V2_CLEAN_MEDICATIONS_PREFIX}{meds_clean_filename}"
logging.info(f"Writing cleaned medications data: {meds_clean_uri}")

start_time = time.time()

df_meds_clean.to_parquet(
    meds_clean_uri,
    engine='pyarrow',
    filesystem=fs,
    compression='snappy',
    index=False
)

elapsed = time.time() - start_time
logging.info(f"Successfully wrote {len(df_meds_clean):,} records in {elapsed:.2f}s")

print(f"✓ Medications clean data written to: {meds_clean_uri}")

2025-11-27 10:04:06,912 INFO Writing cleaned medications data: s3://med-data/v2_clean/medications/medications_clean.parquet
2025-11-27 10:04:06,926 INFO Successfully wrote 36 records in 0.01s


✓ Medications clean data written to: s3://med-data/v2_clean/medications/medications_clean.parquet


---
## Part 5: Verification

In [16]:
# Verify DDI clean data by reading back

logging.info("Verifying DDI clean data...")

start_time = time.time()
df_ddi_verify = pd.read_parquet(ddi_clean_uri, filesystem=fs)
elapsed = time.time() - start_time

assert len(df_ddi_verify) == len(df_ddi_clean), "Row count mismatch!"
assert len(df_ddi_verify.columns) == len(df_ddi_clean.columns), "Column count mismatch!"

logging.info(f"✓ DDI verification successful: {len(df_ddi_verify):,} rows in {elapsed:.2f}s")

print("\nDDI Clean Data (first 5 rows):")
df_ddi_verify.head()

2025-11-27 10:04:15,797 INFO Verifying DDI clean data...
2025-11-27 10:04:15,874 INFO ✓ DDI verification successful: 191,541 rows in 0.08s



DDI Clean Data (first 5 rows):


,Drug 1,Drug 2,Interaction Description,Drug1_Normalized,Drug2_Normalized,Severity
0,Trioxsalen,Verteporfin,Trioxsalen may increase the photosensitizing activities of Verteporfin.,TRIOXSALEN,VERTEPORFIN,Moderate
1,Aminolevulinic acid,Verteporfin,Aminolevulinic acid may increase the photosensitizing activities of Verteporfin.,AMINOLEVULINIC ACID,VERTEPORFIN,Moderate
2,Titanium dioxide,Verteporfin,Titanium dioxide may increase the photosensitizing activities of Verteporfin.,TITANIUM DIOXIDE,VERTEPORFIN,Moderate
3,Tiaprofenic acid,Verteporfin,Tiaprofenic acid may increase the photosensitizing activities of Verteporfin.,TIAPROFENIC ACID,VERTEPORFIN,Moderate
4,Cyamemazine,Verteporfin,Cyamemazine may increase the photosensitizing activities of Verteporfin.,CYAMEMAZINE,VERTEPORFIN,Moderate


In [17]:
# Verify medications clean data by reading back

logging.info("Verifying medications clean data...")

start_time = time.time()
df_meds_verify = pd.read_parquet(meds_clean_uri, filesystem=fs)
elapsed = time.time() - start_time

assert len(df_meds_verify) == len(df_meds_clean), "Row count mismatch!"
assert len(df_meds_verify.columns) == len(df_meds_clean.columns), "Column count mismatch!"

logging.info(f"✓ Medications verification successful: {len(df_meds_verify):,} rows in {elapsed:.2f}s")

print("\nMedications Clean Data (first 5 rows):")
df_meds_verify.head()

2025-11-27 10:04:26,024 INFO Verifying medications clean data...
2025-11-27 10:04:26,050 INFO ✓ Medications verification successful: 36 rows in 0.03s



Medications Clean Data (first 5 rows):


,PatientSID,PatientIEN,Sta3n,DrugNameWithoutDose,DrugNameWithDose,SourceSystem,MedicationDateTime,StartDate,EndDate,Status,...,ControlledSubstanceFlag,OrderNumber,ProviderSID,LocalDrugSID,NationalDrugSID,DrugName_Normalized,MedicationYear,MedicationMonth,MedicationDayOfWeek,MedicationHour
0,1001,PtIEN1001,508,LISINOPRIL,LISINOPRIL 10MG TAB,BCMA,2025-01-01 08:05:00,2025-01-01 08:05:00,NaT,GIVEN,...,None,IP-2025-001001,1001,10002,20002,LISINOPRIL,2025,1,2,8
1,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,BCMA,2025-01-01 12:10:00,2025-01-01 12:10:00,NaT,GIVEN,...,None,IP-2025-001002,1001,10001,20001,METFORMIN,2025,1,2,12
2,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,BCMA,2025-01-01 18:08:00,2025-01-01 18:08:00,NaT,GIVEN,...,None,IP-2025-001002,1001,10001,20001,METFORMIN,2025,1,2,18
3,1001,PtIEN1001,508,LISINOPRIL,LISINOPRIL 10MG TAB,BCMA,2025-01-02 08:45:00,2025-01-02 08:45:00,NaT,GIVEN,...,None,IP-2025-001001,1001,10002,20002,LISINOPRIL,2025,1,3,8
4,1001,PtIEN1001,508,METFORMIN HCL,METFORMIN HCL 500MG TAB,RxOut,2025-01-15 10:30:00,2025-01-15 10:30:00,2025-01-15,ACTIVE,...,N,2024-001-0001,1001,10001,20001,METFORMIN,2025,1,2,10


---
## Part 6: Final Summary

In [18]:
# Final cleaning summary

print("\n" + "="*80)
print("DATA CLEANING SUMMARY")
print("="*80)

print("\nDDI REFERENCE DATASET:")
print(f"  Input:  s3://{DEST_BUCKET}/{V1_RAW_DDI_PREFIX}db_drug_interactions.parquet")
print(f"  Output: s3://{DEST_BUCKET}/{V2_CLEAN_DDI_PREFIX}{ddi_clean_filename}")
print(f"  Records: {len(df_ddi_raw):,} → {len(df_ddi_clean):,}")
print(f"  Columns: {len(df_ddi_raw.columns)} → {len(df_ddi_clean.columns)}")
print(f"  Status: ✓ Complete")

print("\nMEDICATIONS DATASET:")
print(f"  Input:  s3://{DEST_BUCKET}/{V1_RAW_MEDICATIONS_PREFIX}medications_combined.parquet")
print(f"  Output: s3://{DEST_BUCKET}/{V2_CLEAN_MEDICATIONS_PREFIX}{meds_clean_filename}")
print(f"  Records: {len(df_meds_raw):,} → {len(df_meds_clean):,}")
print(f"  Columns: {len(df_meds_raw.columns)} → {len(df_meds_clean.columns)}")
print(f"  Status: ✓ Complete")

print("\nCLEANING OPERATIONS APPLIED:")
print("  ✓ Removed duplicate records")
print("  ✓ Removed records with missing critical values")
print("  ✓ Stripped whitespace from text fields")
print("  ✓ Ensured proper data types")
print("  ✓ Added normalized drug names for matching")
print("  ✓ Added derived columns (severity, date components)")
print("  ✓ Sorted and indexed data")

print("\nNEXT STEPS:")
print("  → Run 04_features.ipynb to engineer features for DDI analysis")

print("="*80)


DATA CLEANING SUMMARY

DDI REFERENCE DATASET:
  Input:  s3://med-data/v1_raw/ddi/db_drug_interactions.parquet
  Output: s3://med-data/v2_clean/ddi/db_drug_interactions_clean.parquet
  Records: 191,541 → 191,541
  Columns: 3 → 6
  Status: ✓ Complete

MEDICATIONS DATASET:
  Input:  s3://med-data/v1_raw/medications/medications_combined.parquet
  Output: s3://med-data/v2_clean/medications/medications_clean.parquet
  Records: 36 → 36
  Columns: 18 → 23
  Status: ✓ Complete

CLEANING OPERATIONS APPLIED:
  ✓ Removed duplicate records
  ✓ Removed records with missing critical values
  ✓ Stripped whitespace from text fields
  ✓ Ensured proper data types
  ✓ Added normalized drug names for matching
  ✓ Added derived columns (severity, date components)
  ✓ Sorted and indexed data

NEXT STEPS:
  → Run 04_features.ipynb to engineer features for DDI analysis
